In [1]:
import lgdo
import lh5
import numpy as np
import awkward as ak
import os
import re

In [16]:
filepath = "/mnt/atlas02/users/leoli/scarf/sp01/data/hit_vcs"
files = os.listdir(filepath)
files.sort()
files

pnum_cal = "00"
run_cal = "037"

pnum_phy_bg = "00"
run_phy_bg = "038"

pnum_phy_signal = "05"
run_phy_signal = "059"

runs = [
    's100ns_mw1', 's100ns_mw3', 's100ns_mw5', 's100ns_mw7',
    's200ns_mw1', 's200ns_mw3', 's200ns_mw5', 's200ns_mw7',
    's60ns_mw1',  's60ns_mw3',  's60ns_mw5',  's60ns_mw7',
    's300ns_mw1', 's300ns_mw3', 's300ns_mw5', 's300ns_mw7','s300ns_mw15',
    's500ns_mw1', 's500ns_mw3', 's500ns_mw5', 's500ns_mw7','s500ns_mw15',
    's700ns_mw1', 's700ns_mw3', 's700ns_mw5', 's700ns_mw7','s700ns_mw15',
    's1000ns_mw1',  's1000ns_mw3',  's1000ns_mw5',  's1000ns_mw7','s1000ns_mw15',
    's2000ns_mw3',  's2000ns_mw5',  's2000ns_mw7',
]

def build_run_db(runs, filepath):
    db = {}
    missing = []

    for run in runs:
        m = re.match(r"s(\d+)ns_mw(\d+)", run)
        if not m:
            missing.append(f"{run}: invalid run name format")
            continue

        run_path = os.path.join(filepath, run)
        phy_bg_path = os.path.join(run_path, "phy", f"p{pnum_phy_bg}", f"r{run_phy_bg}")
        phy_signal_path = os.path.join(run_path, "phy", f"p{pnum_phy_signal}", f"r{run_phy_signal}")
        cal_path = os.path.join(run_path, "cal", f"p{pnum_cal}", f"r{run_cal}")

        missing_this_run = []

        if not os.path.exists(run_path):
            missing_this_run.append(f"run folder missing: {run_path}")
        if not os.path.exists(phy_bg_path):
            missing_this_run.append(f"phy_bg path missing: {phy_bg_path}")
        if not os.path.exists(phy_signal_path):
            missing_this_run.append(f"phy_signal path missing: {phy_signal_path}")
        if not os.path.exists(cal_path):
            missing_this_run.append(f"cal path missing: {cal_path}")

        if missing_this_run:
            missing.append(f"{run}\n  " + "\n  ".join(missing_this_run))
            continue

        phy_bg_files = sorted(os.listdir(phy_bg_path))
        phy_signal_files = sorted(os.listdir(phy_signal_path))
        cal_files = sorted(os.listdir(cal_path))

        db[run] = {
            "run": run,
            "interval": f"{m.group(1)}ns",
            "mw": f"mw{m.group(2)}",

            "phy_bg_path": phy_bg_path,
            "phy_signal_path": phy_signal_path,
            "cal_path": cal_path,

            "phy_bg_files": phy_bg_files,
            "phy_signal_files": phy_signal_files,
            "cal_files": cal_files,

            "phy_bg_pathlist": [os.path.join(phy_bg_path, f) for f in phy_bg_files],
            "phy_signal_pathlist": [os.path.join(phy_signal_path, f) for f in phy_signal_files],
            "cal_pathlist": [os.path.join(cal_path, f) for f in cal_files],
        }

    if missing:
        raise FileNotFoundError(
            "[ERROR] Some runs or paths are missing:\n\n" + "\n\n".join(missing)
        )

    return db
run_db = build_run_db(runs, filepath)

# To start with, lets use 100ns and 5 mw as a pre_alpha_event selection

In [3]:
E_std=lh5.read("ch002/hit/trapEftp_fix_cal",run_db['s100ns_mw5']["phy_signal_pathlist"])
A_max_mw_o_E_Classifier_std=lh5.read("ch002/hit/A_max_mw_o_E_Classifier",run_db['s100ns_mw5']["phy_signal_pathlist"])
Iasy_std = lh5.read("ch002/hit/Iasy",run_db['s100ns_mw5']["phy_signal_pathlist"])
is_valid_waveform_std = lh5.read("ch002/hit/is_valid_waveform",run_db['s100ns_mw5']["phy_signal_pathlist"])
A_max_mw_o_E_High_Side_Cut_std = lh5.read("ch002/hit/A_max_mw_o_E_High_Side_Cut",run_db['s100ns_mw5']["phy_signal_pathlist"])

In [4]:
E_std = np.asarray(E_std)
A_max_mw_o_E_Classifier_std = np.asarray(A_max_mw_o_E_Classifier_std)
Iasy_std = np.asarray(Iasy_std)
is_valid_waveform_std = np.asarray(is_valid_waveform_std)
A_max_mw_o_E_High_Side_Cut_std = np.asarray(A_max_mw_o_E_High_Side_Cut_std)

In [13]:
E_mask = (E_std >= 1839) & (E_std < 2239)
is_not_alpha = (Iasy_std<0) & A_max_mw_o_E_High_Side_Cut_std
ROI_mask = E_mask & is_not_alpha & is_valid_waveform_std

In [6]:
# A_max_mw_o_E_Classifier = {}
# A_max_mw_o_E_Low_Cut = {}
# for run in runs:
#     A_max_mw_o_E_Classifier[run] = lh5.read("ch002/hit/A_max_mw_o_E_Classifier",run_db[run]["phy_signal_pathlist"])
#     A_max_mw_o_E_Low_Cut[run] = lh5.read("ch002/hit/A_max_mw_o_E_Low_Cut",run_db[run]["phy_signal_pathlist"])
    

In [7]:
# for run in runs:
#     A_max_mw_o_E_Classifier[run] = np.asarray(A_max_mw_o_E_Classifier[run])
#     A_max_mw_o_E_Low_Cut[run] = np.asarray(A_max_mw_o_E_Low_Cut[run])

In [8]:
# obj = {
#     "A_max_mw_o_E_Classifier":A_max_mw_o_E_Classifier,
#     'A_max_mw_o_E_Low_Cut':A_max_mw_o_E_Low_Cut,
#     'ROI_mask':ROI_mask
# }
# import pickle
# with open('AOE_Classifier_r059.pkl',"wb") as f:
#     pickle.dump(obj,f)

In [17]:
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
import gc
pickle_dir = Path("AOE_classifier_pickles")
plot_dir = Path("AOE_classifier_plots")
plot_dir.mkdir(parents=True, exist_ok=True)

for run in runs:
    pickle_file = pickle_dir / f"AOE_Classifier_r059_{run}.pkl"

    print(f"Reading {pickle_file}...")

    with open(pickle_file, "rb") as f:
        obj = pickle.load(f)

    aoe = np.asarray(obj["A_max_mw_o_E_Classifier"])
    low_cut = np.asarray(obj["A_max_mw_o_E_Low_Cut"], dtype=bool)
    energy = np.asarray(obj["trapEftp_fix_cal"])

    fig, ax = plt.subplots(figsize=(10, 7))

    ax.hist(
        aoe[ROI_mask],
        bins=300,
        range=(-40, 40),
        histtype="step",
        linewidth=1.5,
        label="A/E Classifier",
    )

    ax.hist(
        aoe[ROI_mask & low_cut],
        bins=300,
        range=(-40, 40),
        histtype="step",
        linewidth=1.5,
        alpha = 0.5,
        color='gray',
        label="low cut accepted",
    )

    ax.set_xlabel("A/E classifier")
    ax.set_ylabel("Counts")
    ax.set_title(f"A/E distribution ROI | r059 | {run}")
    ax.set_xlim(-40, 40)
    ax.set_yscale("log")
    ax.legend()
    ax.grid(alpha=0.2)

    fig.tight_layout()

    output_file = plot_dir / f"AOE_r059_{run}.png"
    fig.savefig(output_file, dpi=200)
    plt.close(fig)

    print(f"Saved {output_file}")

    del obj
    del aoe
    del low_cut
    del energy
    gc.collect()

Reading AOE_classifier_pickles/AOE_Classifier_r059_s100ns_mw1.pkl...
Saved AOE_classifier_plots/AOE_r059_s100ns_mw1.png
Reading AOE_classifier_pickles/AOE_Classifier_r059_s100ns_mw3.pkl...
Saved AOE_classifier_plots/AOE_r059_s100ns_mw3.png
Reading AOE_classifier_pickles/AOE_Classifier_r059_s100ns_mw5.pkl...
Saved AOE_classifier_plots/AOE_r059_s100ns_mw5.png
Reading AOE_classifier_pickles/AOE_Classifier_r059_s100ns_mw7.pkl...
Saved AOE_classifier_plots/AOE_r059_s100ns_mw7.png
Reading AOE_classifier_pickles/AOE_Classifier_r059_s200ns_mw1.pkl...
Saved AOE_classifier_plots/AOE_r059_s200ns_mw1.png
Reading AOE_classifier_pickles/AOE_Classifier_r059_s200ns_mw3.pkl...
Saved AOE_classifier_plots/AOE_r059_s200ns_mw3.png
Reading AOE_classifier_pickles/AOE_Classifier_r059_s200ns_mw5.pkl...
Saved AOE_classifier_plots/AOE_r059_s200ns_mw5.png
Reading AOE_classifier_pickles/AOE_Classifier_r059_s200ns_mw7.pkl...
Saved AOE_classifier_plots/AOE_r059_s200ns_mw7.png
Reading AOE_classifier_pickles/AOE_Class